In [1]:
import json
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.neighbors import NearestNeighbors
import ipywidgets as widgets
from IPython.display import display


class ESBFeaturePlotter:
    """Visualize features with t-SNE, optionally highlighting one category."""

    def __init__(self, feature_dict):
        self.feature_dict = feature_dict

        self.names = list(feature_dict.keys())
        self.features = np.vstack([feature_dict[n]["feature"] for n in self.names])
        self.categories = [feature_dict[n]["category"] for n in self.names]
        self.unique_categories = sorted(set(self.categories))
        self.categories_np = np.array(self.categories)

        perplexity = min(30, max(2, len(self.names) - 1))
        tsne = TSNE(n_components=2, random_state=42, perplexity=perplexity)
        self.features_2d = tsne.fit_transform(self.features)

        # Fit once for NN queries
        self.nbrs = NearestNeighbors(n_neighbors=2, algorithm="auto").fit(self.features)
        self.nn_indices = self.nbrs.kneighbors(self.features, return_distance=False)

    def nn_precision_for_category(self, highlight_category):
        """Precision@1 for queries from one category only."""
        mask = self.categories_np == highlight_category
        query_idx = np.where(mask)[0]

        if len(query_idx) == 0:
            return None

        correct = 0
        for i in query_idx:
            j = self.nn_indices[i, 1]  # nearest neighbor excluding self
            if self.categories_np[j] == highlight_category:
                correct += 1

        return correct / len(query_idx)

    def plot(self, highlight_category=None):
        fig, ax = plt.subplots(figsize=(10, 8))

        if highlight_category is None:
            ax.scatter(
                self.features_2d[:, 0],
                self.features_2d[:, 1],
                c="steelblue",
                s=12,
                alpha=0.75,
                label="All samples",
            )
            title = "t-SNE feature space"
        else:
            other_mask = np.array([c != highlight_category for c in self.categories])
            highlight_mask = ~other_mask

            ax.scatter(
                self.features_2d[other_mask, 0],
                self.features_2d[other_mask, 1],
                c="lightgrey",
                s=10,
                alpha=0.5,
                label="Other",
            )
            ax.scatter(
                self.features_2d[highlight_mask, 0],
                self.features_2d[highlight_mask, 1],
                c="red",
                s=20,
                alpha=0.9,
                label=highlight_category,
            )
            title = f"t-SNE feature space — '{highlight_category}' highlighted"

            # add class-only NN precision text
            p = self.nn_precision_for_category(highlight_category)
            txt = "NN precision: N/A" if p is None else f"NN precision ({highlight_category}): {p:.4f}"
            ax.text(
                0.02,
                0.98,
                txt,
                transform=ax.transAxes,
                va="top",
                ha="left",
                bbox=dict(boxstyle="round", facecolor="white", alpha=0.8),
            )

        ax.legend()
        ax.set_title(title)
        ax.set_xlabel("t-SNE dim 1")
        ax.set_ylabel("t-SNE dim 2")
        plt.tight_layout()
        plt.show()



In [ ]:
JSON_PATH = "experiments/RI_MAE_ESB/SSL_models/pretrain_shapenet/features/ESB_features_100.json"
with open(JSON_PATH, "rb") as f:
    feature_dict = json.load(f)

plotter = ESBFeaturePlotter(feature_dict)

dropdown = widgets.Dropdown(
    options=["(None)"] + plotter.unique_categories,
    value="(None)",
    description="Highlight:",
)

out = widgets.Output()


def render(selected_value):
    selected = None if selected_value == "(None)" else selected_value
    with out:
        out.clear_output(wait=True)  # keeps only one plot visible
        plotter.plot(highlight_category=selected)


def on_change(change):
    if change["name"] == "value":
        render(change["new"])


dropdown.observe(on_change, names="value")
display(widgets.VBox([dropdown, out]))

# initial plot (single visible plot)
render(dropdown.value)